# Solar Filament Segmentation, notebook 01: darkness-threshold baseline and per-pixel logistic regression

COSC 50523 Deep Learning (TCU), course project. Dataset: MAGFiLO (Kaggle competition `filament-segmentation-2026`).

**Task:** predict a binary mask of solar filament pixels for each full-disk solar image, with masks drawn from the COCO polygon annotations.

**Configuration used in this notebook** (read from the config and training cells):

| Setting | Value |
|---|---|
| `IMG_SIZE` | 256 (images and masks resized to 256 x 256) |
| `PATCH` | 9 (9 x 9 neighborhood, 81 features per pixel) |
| `PIX_PER_IMAGE` | 500 pixels per class per training image |
| `SPLIT` | (0.70, 0.15, 0.15) |
| `SPLIT_BY_DAY` | True |
| `SEED` | 42 |
| `CATEGORY_IDS` | None (all annotation categories) |
| Logistic regression | batch gradient descent, `num_iterations=2000`, `learning_rate=0.5` |
| Threshold search on val | probability 0.05 to 0.95 (19 values); darkness cutoff 0.02 to 0.60 (30 values) |

**Split rule:** 70 / 15 / 15 (train / val / test) by whole image *and* by day: the day is the first 8 characters of the file name (YYYYMMDD), and all images from one day go to the same split.

**Metrics:** pooled IoU, Dice, precision and recall over all pixels of the held-out images (TP / FP / FN summed across the split). The decision threshold is tuned on validation (max Dice); the test split is scored once.

**Data is NOT included in this repository.** Download it from Kaggle (competition "Solar Filament Segmentation Challenge 2026", slug `filament-segmentation-2026`) after joining the competition and accepting its rules (otherwise the download returns 403). The download cell below asks for the Kaggle token with `getpass` at runtime; never paste a token into a cell or commit `kaggle.json`.

**Attribution:** the logistic regression helper functions (`sigmoid`, `initialize_with_zeros`, `propagate`, `optimize`) are adapted from our HW2 course assignment. Everything else in this notebook is the team's own code.

# Solar Filament Segmentation: Per-Pixel Logistic Regression (Proof of Concept)

Adapts the HW2 logistic regression to segmentation. Each pixel becomes one training example whose features are the
flattened k x k neighborhood around it, and whose label is the mask value (1 = filament).

Pipeline: load -> split by **whole image** -> balanced pixel sampling (train only) -> HW2 logistic regression ->
full-image prediction -> IoU / Dice / precision / recall -> compare with a simple threshold baseline.

**Data format (MAGFiLO / Kaggle 2026):** `train/train_images/*.jpeg` plus one COCO-style annotation JSON. Masks do not exist as
files, so `load_pairs()` draws them from the polygon annotations. Check the category printout in section 3 and set
`CATEGORY_IDS` if you only want some annotation types.

## 1 - Config and imports

In [ ]:
import os, sys, glob, random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from numpy.lib.stride_tricks import sliding_window_view

SEED = 42
IMG_SIZE = 256        # images are resized to IMG_SIZE x IMG_SIZE (masks use nearest-neighbor)
PATCH = 9             # neighborhood size (odd). 1 = single pixel intensity only
PIX_PER_IMAGE = 500   # pixels sampled per class per training image
SPLIT = (0.70, 0.15, 0.15)  # train / val / test
SPLIT_BY_DAY = True   # keep all images from the same day in one split (neighboring frames are near-duplicates)
CATEGORY_IDS = None   # None = every annotation category; or e.g. [2] after reading the category printout below

DATA_DIR = os.environ.get("DATA_DIR", "/content/filament-segmentation-2026")
_ann = sorted(glob.glob(os.path.join(DATA_DIR, "**", "*Annot*train*.json"), recursive=True))
ANN_PATH = _ann[0] if _ann else None
IMG_DIR = os.path.join(os.path.dirname(ANN_PATH), "train_images") if ANN_PATH else None
print("annotations:", ANN_PATH)
print("images     :", IMG_DIR)

rng = np.random.default_rng(SEED)
random.seed(SEED)

## 2 - Get the data (Colab only)
You must accept the competition rules on the Kaggle competition page first (Join Competition). Then paste your Kaggle API
token when prompted (Kaggle Settings -> API -> generate a token; newer accounts get a token string instead of a `kaggle.json`
file). If you do have a legacy `kaggle.json`, upload it instead when asked. Skip this cell if the data is already in `DATA_DIR`.

In [ ]:
if "google.colab" in sys.modules and not os.path.exists(DATA_DIR):
    import getpass
    os.system("pip install -q --upgrade kaggle")
    tok = getpass.getpass("Paste Kaggle API token (leave empty to upload kaggle.json instead): ").strip()
    if tok:
        os.environ["KAGGLE_API_TOKEN"] = tok
    else:
        from google.colab import files
        files.upload()
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        os.system("mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json")
    os.system("kaggle competitions download -c filament-segmentation-2026 -p /content/dl")
    os.system(f"unzip -q -o /content/dl/*.zip -d {DATA_DIR}")
    _ann = sorted(glob.glob(os.path.join(DATA_DIR, "**", "*Annot*train*.json"), recursive=True))
    ANN_PATH = _ann[0] if _ann else None
    IMG_DIR = os.path.join(os.path.dirname(ANN_PATH), "train_images") if ANN_PATH else None
print(os.path.exists(DATA_DIR), DATA_DIR, ANN_PATH)

## 3 - Inspect the annotations
Prints the annotation categories and how many polygons each has, so you can decide which to use as the filament mask.

In [ ]:
import json
from collections import Counter
A = json.load(open(ANN_PATH))
print("images:", len(A["images"]), " annotations:", len(A["annotations"]))
cnt = Counter(a["category_id"] for a in A["annotations"])
for c in A["categories"]:
    print(c, "-> annotations:", cnt.get(c["id"], 0))
print("example image record:", {k: A["images"][0][k] for k in ("file_name", "height", "width")})

In [ ]:
# Categories found in the COCO JSON and the selection used for the masks
for c in sorted(A["categories"], key=lambda c: c["id"]):
    print(f"category id={c['id']}  name={c['name']!r}  supercategory={c.get('supercategory')!r}")
print("CATEGORY_IDS =", CATEGORY_IDS, " (None means all categories)")

TEAM TO WRITE: which categories correspond to filaments, and whether CATEGORY_IDS should be restricted.

## 4 - Load images and draw masks from the polygons
Each image is decoded as grayscale, resized to `IMG_SIZE`, and scaled to 0-1. Masks are drawn at the same size by scaling the
polygon coordinates; every pixel inside any selected polygon is 1. Images with no annotations get an all-zero mask (valid negatives).

In [ ]:
from PIL import ImageDraw
from collections import defaultdict

def load_pairs(ann_path, img_dir, size=IMG_SIZE, category_ids=CATEGORY_IDS, limit=None):
    A = json.load(open(ann_path))
    by_img = defaultdict(list)
    for a in A["annotations"]:
        if category_ids is None or a["category_id"] in category_ids:
            by_img[a["image_id"]].append(a)
    X, Y, names, skipped_rle = [], [], [], 0
    for rec in A["images"][:limit]:
        p = os.path.join(img_dir, rec["file_name"])
        if not os.path.exists(p): continue
        im = Image.open(p); im.draft("L", (size, size))          # fast JPEG downscale on decode
        im = im.convert("L").resize((size, size), Image.BILINEAR)
        mask = Image.new("L", (size, size), 0); d = ImageDraw.Draw(mask)
        sx, sy = size / rec["width"], size / rec["height"]
        for a in by_img.get(rec["id"], []):
            seg = a["segmentation"]
            if isinstance(seg, dict): skipped_rle += 1; continue   # RLE masks are not handled here
            if seg and not isinstance(seg[0], (list, tuple)): seg = [seg]   # flat polygon -> list of polygons
            for poly in seg:
                pts = np.asarray(poly, dtype=float).reshape(-1, 2)
                if len(pts) >= 3:
                    d.polygon([(x * sx, y * sy) for x, y in pts], fill=255, outline=255)
        a_ = np.asarray(im, dtype=np.float32)
        a_ = (a_ - a_.min()) / (a_.max() - a_.min() + 1e-8)      # per-image min-max to 0-1
        X.append(a_); Y.append((np.asarray(mask) > 0).astype(np.uint8)); names.append(os.path.splitext(rec["file_name"])[0])
    print(f"Loaded {len(X)} images ({len(A['images'])} in JSON); RLE annotations skipped: {skipped_rle}")
    return np.stack(X), np.stack(Y), names

X_all, Y_all, names = load_pairs(ANN_PATH, IMG_DIR)
print("images:", X_all.shape, " masks:", Y_all.shape)
print(f"Filament pixel fraction: {Y_all.mean():.4%}  (this is why accuracy is useless here)")
print(f"Images with no mask pixels: {(Y_all.reshape(len(Y_all), -1).sum(1) == 0).sum()}")

## 5 - Split by whole image (and by day)
Done *before* any pixel sampling so neighboring pixels never leak across splits. File names start with a timestamp (YYYYMMDD...), and frames from the same day look almost identical, so by default whole days go to one split.

In [ ]:
n = len(X_all)
days = np.array([nm[:8] for nm in names])
if SPLIT_BY_DAY and all(d.isdigit() for d in days):
    ud = rng.permutation(np.unique(days))
    cum = np.cumsum([(days == d).sum() for d in ud])
    k1, k2 = np.searchsorted(cum, SPLIT[0] * n) + 1, np.searchsorted(cum, (SPLIT[0] + SPLIT[1]) * n) + 1
    which = {d: 0 for d in ud[:k1]} | {d: 1 for d in ud[k1:k2]} | {d: 2 for d in ud[k2:]}
    grp = np.array([which[d] for d in days])
    idx_tr, idx_va, idx_te = (np.flatnonzero(grp == g) for g in range(3))
else:
    perm = rng.permutation(n)
    n_tr, n_va = int(SPLIT[0] * n), int(SPLIT[1] * n)
    idx_tr, idx_va, idx_te = perm[:n_tr], perm[n_tr:n_tr + n_va], perm[n_tr + n_va:]
print(f"train={len(idx_tr)}  val={len(idx_va)}  test={len(idx_te)} images")

# Quick look at a few pairs
fig, ax = plt.subplots(2, 4, figsize=(14, 7))
for j, i in enumerate(idx_tr[:4]):
    ax[0, j].imshow(X_all[i], cmap="gray"); ax[0, j].set_title(names[i]); ax[0, j].axis("off")
    ax[1, j].imshow(Y_all[i], cmap="gray"); ax[1, j].axis("off")
plt.show()

## Dataset statistics and leakage checks
Per-split counts computed from the split indices above (no masks are recomputed), followed by two leakage assertions.

In [ ]:
# Dataset statistics and leakage checks (uses names, days, Y_all, A and idx_tr / idx_va / idx_te from above)
# image records in the same order as load_pairs() (it skips records whose file is missing); some file names repeat
_recs = [r for r in A["images"] if os.path.exists(os.path.join(IMG_DIR, r["file_name"]))]
assert len(_recs) == len(names), (len(_recs), len(names))
_ann_per_img = Counter(a["image_id"] for a in A["annotations"]
                       if CATEGORY_IDS is None or a["category_id"] in CATEGORY_IDS)
_fmt_day = lambda d: f"{d[:4]}-{d[4:6]}-{d[6:8]}" if d.isdigit() else d
split_days, split_names = {}, {}
for split_name, idx in (("train", idx_tr), ("val", idx_va), ("test", idx_te)):
    d = sorted(set(days[idx]))
    split_days[split_name], split_names[split_name] = set(d), {names[i] for i in idx}
    n_px = len(idx) * Y_all[0].size
    frac = sum(int(Y_all[i].sum()) for i in idx) / n_px if n_px else float("nan")
    n_ann = sum(_ann_per_img.get(_recs[i]["id"], 0) for i in idx)
    print(f"{split_name:5s}: images={len(idx):5d}  days={len(d):4d}  "
          f"first={_fmt_day(d[0]) if d else '-'}  last={_fmt_day(d[-1]) if d else '-'}  "
          f"filament pixel fraction={frac:.4%}  annotations={n_ann}")
_name_counts = Counter(names)
print(f"annotations total over splits = {sum(_ann_per_img.get(r['id'], 0) for r in _recs)}  "
      f"(selected annotations in JSON = {sum(_ann_per_img.values())})")
print(f"file names used by more than one image record: {sum(c > 1 for c in _name_counts.values())} "
      f"({sum(c - 1 for c in _name_counts.values())} extra records)")

_pairs = [("train", "val"), ("train", "test"), ("val", "test")]
day_overlap = {f"{a}/{b}": sorted(split_days[a] & split_days[b]) for a, b in _pairs}
name_overlap = {f"{a}/{b}": sorted(split_names[a] & split_names[b]) for a, b in _pairs}
n_day_overlap = sum(len(v) for v in day_overlap.values())
n_name_overlap = sum(len(v) for v in name_overlap.values())
print(f"Check 1 - days in more than one split: {n_day_overlap}", {k: len(v) for k, v in day_overlap.items()})
print(f"Check 2 - file names in more than one split: {n_name_overlap}", {k: len(v) for k, v in name_overlap.items()})
assert n_name_overlap == 0, f"file names shared between splits: {name_overlap}"
if SPLIT_BY_DAY:
    assert n_day_overlap == 0, f"days shared between splits (day split did not run?): {day_overlap}"
else:
    print("SPLIT_BY_DAY is False, so check 1 is reported but not asserted")
print("Leakage checks passed")

## 6 - Patch features and balanced pixel sampling
`patch_features(img)` returns every pixel's k x k neighborhood as a column, shape `(PATCH*PATCH, H*W)`, matching the HW2
convention of `(features, examples)`. Training samples an equal number of filament and background pixels per image
(background is subsampled; evaluation later uses all pixels).

In [ ]:
def patch_features(img, patch=PATCH):
    r = patch // 2
    padded = np.pad(img, r, mode="reflect")
    win = sliding_window_view(padded, (patch, patch))        # (H, W, patch, patch)
    return win.reshape(-1, patch * patch).T.astype(np.float32)  # (patch*patch, H*W)

def sample_pixels(X, Y, idxs, per_class=PIX_PER_IMAGE, patch=PATCH):
    feats, labels = [], []
    for i in idxs:
        F = patch_features(X[i], patch)
        y = Y[i].reshape(-1)
        pos, neg = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
        if len(pos) == 0:  # skip images without filaments for the positive class, still take negatives
            take = rng.choice(neg, size=min(per_class, len(neg)), replace=False)
        else:
            take = np.concatenate([
                rng.choice(pos, size=min(per_class, len(pos)), replace=False),
                rng.choice(neg, size=min(per_class, len(neg)), replace=False)])
        feats.append(F[:, take]); labels.append(y[take])
    return np.concatenate(feats, axis=1), np.concatenate(labels)[None, :].astype(np.float32)

train_set_x, train_set_y = sample_pixels(X_all, Y_all, idx_tr)
print("train_set_x:", train_set_x.shape, " train_set_y:", train_set_y.shape,
      f" positives: {train_set_y.mean():.1%}")

## 7 - HW2 logistic regression
Same functions as HW2 (sigmoid, init, propagate, optimize, predict). The only change is an epsilon clip inside the log so a saturated sigmoid can't produce `-inf`.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def initialize_with_zeros(dim):
    return np.zeros((dim, 1)), 0.0

def propagate(w, b, X, Y):
    m = X.shape[1]
    A = sigmoid(np.dot(w.T, X) + b)
    eps = 1e-12
    cost = -1 / m * (np.dot(Y, np.log(A + eps).T) + np.dot(1 - Y, np.log(1 - A + eps).T))
    dw = 1 / m * np.dot(X, (A - Y).T)
    db = 1 / m * np.sum(A - Y)
    return {"dw": dw, "db": db}, np.squeeze(np.array(cost))

def optimize(w, b, X, Y, num_iterations=2000, learning_rate=0.5, print_cost=False):
    costs = []
    for i in range(num_iterations):
        grads, cost = propagate(w, b, X, Y)
        w = w - learning_rate * grads["dw"]
        b = b - learning_rate * grads["db"]
        if i % 100 == 0:
            costs.append(cost)
            if print_cost: print(f"Cost after iteration {i}: {cost:.6f}")
    return {"w": w, "b": b}, grads, costs

def predict_proba(w, b, X):
    return sigmoid(np.dot(w.T, X) + b)

In [ ]:
w, b = initialize_with_zeros(train_set_x.shape[0])
params, grads, costs = optimize(w, b, train_set_x, train_set_y, num_iterations=2000, learning_rate=0.5, print_cost=True)
w, b = params["w"], params["b"]
plt.plot(np.squeeze(costs)); plt.xlabel("iterations (hundreds)"); plt.ylabel("cost"); plt.show()

## 8 - Metrics
Computed on **all** pixels of each held-out image, so the true class imbalance is preserved. IoU/Dice are
pooled over the whole split (sum of TP/FP/FN across images) rather than averaged per image, which avoids
undefined values on images with no filaments.

In [ ]:
def confusion(pred, true):
    pred, true = pred.astype(bool), true.astype(bool)
    return (np.sum(pred & true), np.sum(pred & ~true), np.sum(~pred & true))  # TP, FP, FN

def scores(tp, fp, fn, eps=1e-8):
    return {"IoU": tp / (tp + fp + fn + eps),
            "Dice": 2 * tp / (2 * tp + fp + fn + eps),
            "Precision": tp / (tp + fp + eps),
            "Recall": tp / (tp + fn + eps)}

def predict_masks_lr(X, idxs, thr=0.5):
    probs = [predict_proba(w, b, patch_features(X[i])).reshape(IMG_SIZE, IMG_SIZE) for i in idxs]
    return np.stack(probs)

def evaluate(prob_maps, Y, idxs, thr):
    tp = fp = fn = 0
    for p, i in zip(prob_maps, idxs):
        a, b_, c = confusion(p >= thr, Y[i]); tp += a; fp += b_; fn += c
    return scores(tp, fp, fn)

# Tune the decision threshold on the validation images (maximize Dice), then score the test images once.
val_probs = predict_masks_lr(X_all, idx_va)
thr_grid = np.linspace(0.05, 0.95, 19)
best_thr = max(thr_grid, key=lambda t: evaluate(val_probs, Y_all, idx_va, t)["Dice"])
print(f"Best threshold on val: {best_thr:.2f}")
test_probs = predict_masks_lr(X_all, idx_te)
lr_val = evaluate(val_probs, Y_all, idx_va, best_thr)
lr_test = evaluate(test_probs, Y_all, idx_te, best_thr)
print("LogReg  val :", {k: round(float(v), 4) for k, v in lr_val.items()})
print("LogReg  test:", {k: round(float(v), 4) for k, v in lr_test.items()})

## 9 - Non-learning baseline: darkness threshold
Filaments are dark, so flag pixels darker than a cutoff. The cutoff is tuned on the validation images with the same metric.

In [ ]:
def thresh_masks(X, idxs, cut):
    return np.stack([(X[i] < cut) for i in idxs])

def eval_thresh(idxs, cut):
    tp = fp = fn = 0
    for m, i in zip(thresh_masks(X_all, idxs, cut), idxs):
        a, b_, c = confusion(m, Y_all[i]); tp += a; fp += b_; fn += c
    return scores(tp, fp, fn)

cut_grid = np.linspace(0.02, 0.6, 30)
best_cut = max(cut_grid, key=lambda c: eval_thresh(idx_va, c)["Dice"])
th_test = eval_thresh(idx_te, best_cut)
print(f"Best darkness cutoff on val: {best_cut:.2f}")
print("Threshold test:", {k: round(float(v), 4) for k, v in th_test.items()})

## 10 - Results table

In [ ]:
import pandas as pd
res = pd.DataFrame({"Threshold baseline": th_test, "Logistic regression (per-pixel)": lr_test}).T.round(4)
display(res)

In [ ]:
# Save the test-split numbers so 02_unet.ipynb can read them (outputs/ is git-ignored)
OUTPUT_DIR = os.environ.get("OUTPUT_DIR", "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)
_out = os.path.join(OUTPUT_DIR, "results_baselines.json")
with open(_out, "w") as f:
    json.dump({"img_size": IMG_SIZE, "split": "test", "seed": SEED,
               "Threshold baseline": {k: float(v) for k, v in th_test.items()},
               "Logistic regression (per-pixel)": {k: float(v) for k, v in lr_test.items()}}, f, indent=2)
print("wrote", _out)

## 11 - Visualize predictions
Image, ground truth, logistic regression probability, thresholded prediction. Look for failure modes such as sunspots or the solar limb being flagged as filament.

In [ ]:
k = min(4, len(idx_te))
fig, ax = plt.subplots(k, 4, figsize=(14, 3.5 * k), squeeze=False)
for r in range(k):
    i = idx_te[r]
    ax[r, 0].imshow(X_all[i], cmap="gray"); ax[r, 0].set_title("image")
    ax[r, 1].imshow(Y_all[i], cmap="gray"); ax[r, 1].set_title("ground truth")
    ax[r, 2].imshow(test_probs[r], cmap="viridis", vmin=0, vmax=1); ax[r, 2].set_title("LR probability")
    ax[r, 3].imshow(test_probs[r] >= best_thr, cmap="gray"); ax[r, 3].set_title(f"LR mask (thr={best_thr:.2f})")
    for a in ax[r]: a.axis("off")
plt.tight_layout(); plt.show()

## Next steps
- Try `PATCH` of 1, 5, 9, 15 and see how much context helps.
- Add simple extra features (local mean/std, gradient magnitude) as extra rows of `train_set_x`.
- Reuse sections 3-5, 8, 9, 10, and 11 unchanged for the U-Net: only the model and training loop change.

## Planned work (TEAM TO CONFIRM)
- Full-resolution / crop-based training (requested by the professor)
- A second architecture (requested by the professor)
- A hyperparameter comparison (requested by the professor)
- A disc mask for the darkness-threshold baseline